# Additional MSWEGNN metrics

This notebook post-processes the `.npz` files produced by `train.py --with_test True`. It calculates depth MAE, discharge MAE, optional unit-discharge MAE, CSI at 0.05 m and 0.30 m, and inference-time mean/standard deviation across events.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

from data_mswegnn.shp_data_retrieval import get_cell_area, get_face_length, get_node_types


In [ ]:
# Adjust these paths if your results are stored elsewhere.
PROJECT_ROOT = Path.cwd()
# METRICS_DIR = PROJECT_ROOT / 'saved_metrics'
METRICS_DIR = PROJECT_ROOT / 'saved_metrics'
RAW_DIR = PROJECT_ROOT / 'data_mswegnn' / 'datasets' / 'raw'
TEST_CSV = RAW_DIR / 'test.csv'
OUTPUT_CSV = PROJECT_ROOT / 'additional_mswegnn_metrics.csv'

# Use one model run, or set to None to process every matching file.
MODEL_PREFIX = "DUALFloodGNN_2026-09-27_08-17-01"  # e.g. 'DUALFloodGNN_2026-09-27_12-30-00'
UNIT_DISCHARGE = True


In [ ]:
summary = pd.read_csv(TEST_CSV)
summary['Run_ID'] = summary['Run_ID'].astype(str)
summary_by_run = summary.set_index('Run_ID')

def relative_raw_path(value):
    return RAW_DIR / str(value).replace('/', '\\')

def csi(pred_depth, target_depth, threshold):
    pred_flooded = pred_depth > threshold
    target_flooded = target_depth > threshold
    tp = np.logical_and(pred_flooded, target_flooded).sum()
    fp = np.logical_and(pred_flooded, ~target_flooded).sum()
    fn = np.logical_and(~pred_flooded, target_flooded).sum()
    denominator = tp + fp + fn
    return float(tp / denominator) if denominator else np.nan

def mean_sd(values):
    values = np.asarray(values, dtype=float)
    return float(np.mean(values)), float(np.std(values, ddof=1)) if len(values) > 1 else 0.0


In [ ]:
files = sorted(METRICS_DIR.glob('*_runid_*_test_metrics.npz'))
if MODEL_PREFIX is not None:
    files = [path for path in files if path.name.startswith(MODEL_PREFIX + '_runid_')]
if not files:
    raise FileNotFoundError(f'No metric files found in {METRICS_DIR}')

rows = []
for path in files:
    match = re.search(r'_runid_(.+?)_test_metrics\.npz$', path.name)
    if match is None:
        continue
    run_id = match.group(1)
    event = summary_by_run.loc[run_id]
    data = np.load(path, allow_pickle=True)

    # Saved node arrays already exclude boundary nodes, matching the tester.
    cell_area_all = get_cell_area(relative_raw_path(event['Cells_Shp_Filepath']))
    node_types = get_node_types(relative_raw_path(event['Nodes_Shp_Filepath']))
    cell_area = cell_area_all[node_types != 0]
    pred_volume = np.asarray(data['pred'], dtype=float)
    target_volume = np.asarray(data['target'], dtype=float)
    if pred_volume.shape[-1] != len(cell_area):
        raise ValueError(f'{path.name}: {pred_volume.shape} does not match {len(cell_area)} non-boundary cell areas')
    pred_depth = pred_volume / cell_area
    target_depth = target_volume / cell_area

    edge_pred = np.asarray(data['edge_pred'], dtype=float)
    edge_target = np.asarray(data['edge_target'], dtype=float)
    edge_mae = np.mean(np.abs(edge_pred - edge_target))
    edge_length = get_face_length(relative_raw_path(event['Edges_Shp_Filepath']))
    unit_discharge_mae = np.mean(np.abs(edge_pred / edge_length - edge_target / edge_length))

    rows.append({
        'run_id': run_id,
        'source_file': path.name,
        'depth_mae_m': np.mean(np.abs(pred_depth - target_depth)),
        'discharge_mae_m3_per_s': edge_mae,
        'unit_discharge_mae_m2_per_s': unit_discharge_mae if UNIT_DISCHARGE else np.nan,
        'csi_005m': csi(pred_depth, target_depth, 0.05),
        'csi_030m': csi(pred_depth, target_depth, 0.30),
        'inference_time_s_per_timestep': float(data['inference_time']),
    })

results = pd.DataFrame(rows).sort_values('run_id').reset_index(drop=True)
results.to_csv(OUTPUT_CSV, index=False)
results


In [ ]:
metric_columns = [
    'depth_mae_m', 'discharge_mae_m3_per_s',
    'unit_discharge_mae_m2_per_s', 'csi_005m', 'csi_030m',
    'inference_time_s_per_timestep'
]
summary_stats = pd.DataFrame({
    'mean': results[metric_columns].mean(),
    'sd': results[metric_columns].std(ddof=1).fillna(0.0),
})
summary_stats
